1. Producteur de logs :

In [ ]:
from kafka import KafkaProducer
import logging
import time
import random

# Configuration Kafka
producer = KafkaProducer(bootstrap_servers='localhost:9092')
topic_name = 'logs-topic'

# Types de logs simulés
log_types = ['INFO', 'WARNING', 'ERROR', 'DEBUG']
messages = [
    "User logged in", "Failed login attempt", "Database connection established",
    "Server started on port 8080", "Disk space low", "404 Not Found",
    "Invalid request received", "Backup completed successfully"
]

def generate_log():
    while True:
        log_type = random.choice(log_types)
        message = random.choice(messages)
        log_entry = f"{time.ctime()} [{log_type}] {message}"
        
        # Envoi à Kafka
        producer.send(topic_name, log_entry.encode('utf-8'))
        print(f"Sent: {log_entry}")
        
        # Intervalle aléatoire entre 0.1 et 1 seconde
        time.sleep(random.uniform(0.1, 1))

if __name__ == "__main__":
    generate_log()

2. Consumer Spark : 

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("RealTimeLogProcessor") \
    .master("spark://<VOTRE_IP>:7077") \
    .config("spark.executor.instances", "2") \
    .config("spark.executor.cores", "2") \
    .getOrCreate()

# Schéma pour les logs
log_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("log_level", StringType(), True),
    StructField("message", StringType(), True)
])

# Lecture du flux Kafka
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "logs-topic") \
    .option("startingOffsets", "latest") \
    .load()

# Transformation des données
logs_df = df.selectExpr("CAST(value AS STRING)") \
    .withColumn("extracted", regexp_extract(col("value"), r'^(.*?) \[(.*?)\] (.*)$', 0)) \
    .select(
        regexp_extract(col("value"), r'^(.*?) \[(.*?)\] (.*)$', 1).alias("timestamp"),
        regexp_extract(col("value"), r'^(.*?) \[(.*?)\] (.*)$', 2).alias("log_level"),
        regexp_extract(col("value"), r'^(.*?) \[(.*?)\] (.*)$', 3).alias("message")
    )

# Statistiques en temps réel
stats_df = logs_df.groupBy("log_level").count()

# Affichage des résultats
query = stats_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

query.awaitTermination()

3. Classification des logs : 

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

# Configuration de la session Spark
spark = SparkSession.builder \
    .appName("RealTimeLogClassifier") \
    .master("spark://<VOTRE_IP>:7077") \
    .config("spark.executor.instances", "2") \
    .config("spark.executor.cores", "2") \
    .config("spark.dynamicAllocation.enabled", "false") \
    .config("spark.streaming.kafka.maxRatePerPartition", "1000") \
    .getOrCreate()

# Schéma pour les logs entrants
log_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("log_level", StringType(), True),
    StructField("message", StringType(), True)
])

# Configuration Kafka
kafka_params = {
    "kafka.bootstrap.servers": "localhost:9092",
    "subscribe": "logs-topic",
    "startingOffsets": "latest"
}

# Lecture du flux Kafka
raw_logs = spark.readStream \
    .format("kafka") \
    .options(**kafka_params) \
    .load()

# Transformation des données
parsed_logs = raw_logs.select(
    from_json(col("value").cast("string"), log_schema).alias("data")) \
    .select("data.*")

# Classification des logs
classified_logs = parsed_logs.withColumn("log_category", 
    when(col("log_level") == "ERROR", "Critical")
    .when(col("log_level") == "WARNING", "Warning")
    .when(col("log_level").isin(["INFO", "DEBUG"]), "Information")
    .otherwise("Other"))

# Statistiques en temps réel
stats = classified_logs.groupBy(
    "log_level", 
    "log_category"
).count().orderBy("count", ascending=False)

# Fonction pour écrire les résultats
def write_to_console(df, epoch_id):
    print(f"\nBatch {epoch_id} - {time.strftime('%Y-%m-%d %H:%M:%S')}")
    df.show(truncate=False)

# Fonction pour écrire dans HDFS (optionnel)
def write_to_hdfs(df, epoch_id):
    df.write \
        .mode("append") \
        .parquet("hdfs://localhost:9000/logs/classified")

# Lancement du traitement
query = stats.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_to_console) \
    .option("truncate", "false") \
    .start()

# Option: Ecriture dans HDFS
# hdfs_query = classified_logs.writeStream \
#     .outputMode("append") \
#     .foreachBatch(write_to_hdfs) \
#     .option("checkpointLocation", "/tmp/checkpoint_logs") \
#     .start()

query.awaitTermination()